## Split manifest — the 117 groups of the locked hold-out set

Writes `holdout_groups.txt`.

The draw is over **group names**, stratified by annotation content into
four strata (background only, MILCO only, NOMBO only, both present),
under `random.Random(42)`. Nothing about the image pixels enters it, so
this reproduces the same 117 groups on any machine from the published
dataset alone.

Expected output: `1170 pairs`, strata `866 / 181 / 74 / 49`, `117 groups`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/santos_sss.zip'
RAW = '/content/santos_raw'
if not os.path.isdir(RAW):
    zipfile.ZipFile(ZIP).extractall(RAW)
print('dataset ready at', RAW)

In [ ]:
import os, glob, random

RAW  = '/content/santos_raw'
SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

pairs = []
for ip in glob.glob(f'{RAW}/**/*', recursive=True):
    if os.path.splitext(ip)[1].lower() not in IMG_EXT:
        continue
    lp = os.path.splitext(ip)[0] + '.txt'
    if os.path.exists(lp):
        pairs.append((os.path.splitext(os.path.basename(ip))[0], lp))
pairs.sort()
print(f'image/label pairs found: {len(pairs)}  (expected 1170)')

def stratum(lp):
    cs = set()
    for line in open(lp):
        p = line.split()
        if len(p) >= 5:
            cs.add(int(float(p[0])))
    if not cs:      return 'bg'
    if cs == {0}:   return 'milco'
    if cs == {1}:   return 'nombo'
    return 'both'

strata = {stem: stratum(lp) for stem, lp in pairs}
counts = {k: sum(v == k for v in strata.values()) for k in ('bg','milco','both','nombo')}
print('strata:', counts, ' (expected bg 866, milco 181, both 74, nombo 49)')

rng, bys = random.Random(SEED), {}
for stem, _ in pairs:
    bys.setdefault(strata[stem], []).append(stem)

hold = []
for k in sorted(bys):
    g = sorted(bys[k])
    rng.shuffle(g)
    hold += g[:round(len(g) * 0.10)]

hold.sort()
assert len(hold) == 117, f'expected 117 hold-out groups, got {len(hold)}'
with open('holdout_groups.txt', 'w') as f:
    f.write('\n'.join(hold) + '\n')
print(f'\nholdout_groups.txt written: {len(hold)} groups '
      f'= {len(hold)*4} files after 4x augmentation')
print(f'cross-validation pool: {len(pairs)-len(hold)} groups '
      f'= {(len(pairs)-len(hold))*4} files')

from google.colab import files
files.download('holdout_groups.txt')